In [1]:
#1
# Check GPU.
!nvidia-smi

# Install required packages.
!pip -q install -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" accelerate tqdm numpy psutil

import sys
import torch
import importlib.metadata as md

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    raise RuntimeError("CUDA is not available. Please switch Colab runtime to GPU.")

print("transformers:", md.version("transformers"))
print("sentence-transformers:", md.version("sentence-transformers"))

Fri May 22 19:21:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
#2
# Mount Google Drive and define paths.
from google.colab import drive
from pathlib import Path
import shutil
import os
import json

drive.mount("/content/drive")

DATASET = "hotpotqa"

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
IDEA_DIR = PROJECT_DIR / "idea_1"
DATASET_DIR = IDEA_DIR / "kg" / DATASET

CLEAN_INPUT_PATH = DATASET_DIR / "hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json"

# Prefer the normalized KG JSONL created in the entity notebook.
NORMALIZED_KG_JSONL_PATH = DATASET_DIR / "kg_construct" / "hotpotqa_kg_extractions.clean.normalized_entities.jsonl"

# Output folder on Drive.
CONSTRUCT_DIR = DATASET_DIR / "kg_construct"
CONSTRUCT_DIR.mkdir(parents=True, exist_ok=True)

# Local work folder for faster I/O.
LOCAL_WORK_DIR = Path(f"/content/kg_construct_work/{DATASET}")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_INPUT_PATH.exists():
    raise FileNotFoundError(f"Clean input file not found: {CLEAN_INPUT_PATH}")

USE_NORMALIZED_JSONL = NORMALIZED_KG_JSONL_PATH.exists()

print("Clean input:", CLEAN_INPUT_PATH)
print("Normalized KG JSONL exists:", USE_NORMALIZED_JSONL)
print("Normalized KG JSONL:", NORMALIZED_KG_JSONL_PATH)
print("Drive output folder:", CONSTRUCT_DIR)
print("Local work folder:", LOCAL_WORK_DIR)

Mounted at /content/drive
Clean input: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json
Normalized KG JSONL exists: True
Normalized KG JSONL: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_kg_extractions.clean.normalized_entities.jsonl
Drive output folder: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct
Local work folder: /content/kg_construct_work/hotpotqa


In [3]:
#3
# Main configuration.
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

EMBED_BATCH_SIZE = 32
MAX_SEQ_LENGTH = 128
EMBEDDING_DIM = 4096

# Primary requested outputs.
RELATION_EMBEDDINGS_NPY_LOCAL = LOCAL_WORK_DIR / "relation_embeddings.npy"
FACT_EMBEDDINGS_NPY_LOCAL = LOCAL_WORK_DIR / "fact_embeddings.npy"

RELATION_ID_TO_ROW_JSONL_LOCAL = LOCAL_WORK_DIR / "relation_id_to_row.jsonl"
FACT_ID_TO_ROW_JSONL_LOCAL = LOCAL_WORK_DIR / "fact_id_to_row.jsonl"

RELATION_EMBEDDINGS_NPY_DRIVE = CONSTRUCT_DIR / RELATION_EMBEDDINGS_NPY_LOCAL.name
FACT_EMBEDDINGS_NPY_DRIVE = CONSTRUCT_DIR / FACT_EMBEDDINGS_NPY_LOCAL.name

RELATION_ID_TO_ROW_JSONL_DRIVE = CONSTRUCT_DIR / RELATION_ID_TO_ROW_JSONL_LOCAL.name
FACT_ID_TO_ROW_JSONL_DRIVE = CONSTRUCT_DIR / FACT_ID_TO_ROW_JSONL_LOCAL.name

print("Model:", MODEL_NAME)
print("Embedding batch size:", EMBED_BATCH_SIZE)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Expected embedding dim:", EMBEDDING_DIM)

print("\nRelation outputs:")
print(RELATION_EMBEDDINGS_NPY_DRIVE)
print(RELATION_ID_TO_ROW_JSONL_DRIVE)

print("\nFact outputs:")
print(FACT_EMBEDDINGS_NPY_DRIVE)
print(FACT_ID_TO_ROW_JSONL_DRIVE)

Model: Qwen/Qwen3-Embedding-8B
Embedding batch size: 32
Max sequence length: 128
Expected embedding dim: 4096

Relation outputs:
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_embeddings.npy
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_id_to_row.jsonl

Fact outputs:
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_embeddings.npy
/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_id_to_row.jsonl


In [4]:
#4
# Helper functions.
import json
import os
import re
import html
import unicodedata
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np

def atomic_json_dump(obj, path):
    # Write JSON safely.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def normalize_entity_text(text):
    # Match the entity-normalization style used earlier.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" \t\r\n\"'`")
    text = text.lower()

    return text

def normalize_edge_text(text):
    # Clean relation/info text without lowercasing.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def count_jsonl_lines(path):
    # Count JSONL rows.
    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for _ in f:
            count += 1

    return count

def iter_kg_items():
    # Prefer normalized JSONL if available.
    if USE_NORMALIZED_JSONL:
        with open(NORMALIZED_KG_JSONL_PATH, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    yield json.loads(line)
    else:
        with open(CLEAN_INPUT_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            raise RuntimeError("Clean input JSON must be a list.")

        for input_index, item in enumerate(data):
            item = dict(item)
            item["input_index"] = item.get("input_index", input_index)
            item["chunk_id"] = item.get("chunk_id") or item.get("Chunk_id")
            item["title"] = item.get("title") or item.get("Title")
            item["paragraph_id"] = item.get("paragraph_id") or item.get("Paragraph_id")
            item["token_count"] = item.get("token_count") or item.get("Token_count")
            yield item

def copy_output_to_drive(local_path, drive_path):
    # Copy one output file to Google Drive.
    local_path = Path(local_path)
    drive_path = Path(drive_path)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local output: {local_path}")

    shutil.copy2(local_path, drive_path)
    print("Copied:", drive_path)

def verify_embedding_output(catalog_path, embeddings_path, id_key):
    # Verify mapping rows and embedding shape.
    catalog_path = Path(catalog_path)
    embeddings_path = Path(embeddings_path)

    n_rows = count_jsonl_lines(catalog_path)
    emb = np.load(embeddings_path, mmap_mode="r")

    print("Catalog rows:", n_rows)
    print("Embedding shape:", emb.shape)
    print("Embedding dtype:", emb.dtype)

    if emb.shape != (n_rows, EMBEDDING_DIM):
        raise RuntimeError(f"Bad embedding shape: {emb.shape} != {(n_rows, EMBEDDING_DIM)}")

    if emb.dtype != np.float32:
        raise RuntimeError(f"Bad embedding dtype: {emb.dtype}")

    # Check first and last catalog rows.
    first = None
    last = None

    with open(catalog_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                if first is None:
                    first = rec
                last = rec

    if n_rows > 0:
        if first[id_key] != 0 or first["row"] != 0:
            raise RuntimeError("First mapping row is invalid.")

        if last[id_key] != n_rows - 1 or last["row"] != n_rows - 1:
            raise RuntimeError("Last mapping row is invalid.")

        sample_count = min(1000, n_rows)
        sample_indices = np.linspace(0, n_rows - 1, sample_count, dtype=np.int64)
        sample = np.asarray(emb[sample_indices])
        norms = np.linalg.norm(sample, axis=1)

        print("Norm min/mean/max:", float(norms.min()), float(norms.mean()), float(norms.max()))

    print("Verification passed.")

In [5]:
#5
# Load Qwen3 embedding model on GPU.
import os
import torch
from sentence_transformers import SentenceTransformer

os.environ["HF_HOME"] = str(LOCAL_WORK_DIR / "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_WORK_DIR / "hf_cache" / "transformers")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
    model_kwargs={
        "device_map": "auto",
        "attn_implementation": "sdpa",
    },
    tokenizer_kwargs={
        "padding_side": "left",
    },
)

model.max_seq_length = MAX_SEQ_LENGTH

print("Loaded model:", MODEL_NAME)
print("Max sequence length:", model.max_seq_length)
print("Device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-Embedding-8B
Max sequence length: 128
Device: cuda:0


In [6]:
#6
# Stream JSONL texts into a memory-mapped .npy embedding file.
import gc
import torch
import numpy as np
from tqdm.auto import tqdm

def encode_catalog_jsonl_to_npy(
    catalog_jsonl_path,
    embeddings_npy_path,
    text_key,
    expected_dim=4096,
    batch_size=32,
    flush_every_batches=50,
    overwrite=True,
):
    catalog_jsonl_path = Path(catalog_jsonl_path)
    embeddings_npy_path = Path(embeddings_npy_path)

    if not catalog_jsonl_path.exists():
        raise FileNotFoundError(f"Catalog not found: {catalog_jsonl_path}")

    total_rows = count_jsonl_lines(catalog_jsonl_path)

    if total_rows == 0:
        raise RuntimeError(f"No rows found in catalog: {catalog_jsonl_path}")

    if embeddings_npy_path.exists() and not overwrite:
        existing = np.load(embeddings_npy_path, mmap_mode="r")
        if existing.shape == (total_rows, expected_dim) and existing.dtype == np.float32:
            print("Embedding file already exists and looks valid. Skipping:", embeddings_npy_path)
            return
        raise RuntimeError(
            f"Existing embedding file is invalid: shape={existing.shape}, dtype={existing.dtype}"
        )

    embeddings_memmap = np.lib.format.open_memmap(
        embeddings_npy_path,
        mode="w+",
        dtype=np.float32,
        shape=(total_rows, expected_dim),
    )

    row_cursor = 0
    batch_texts = []
    batch_start_row = 0
    batch_counter = 0

    def write_batch(texts, start_row):
        nonlocal batch_counter

        batch_embeddings = model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        batch_embeddings = np.asarray(batch_embeddings)

        if batch_embeddings.dtype != np.float32:
            raise RuntimeError(
                f"Expected float32 embeddings, got {batch_embeddings.dtype}. "
                "No automatic conversion is done."
            )

        if batch_embeddings.shape[1] != expected_dim:
            raise RuntimeError(
                f"Unexpected embedding dim: {batch_embeddings.shape[1]} != {expected_dim}"
            )

        end_row = start_row + len(texts)
        embeddings_memmap[start_row:end_row, :] = batch_embeddings

        batch_counter += 1

        if batch_counter % flush_every_batches == 0:
            embeddings_memmap.flush()

    with open(catalog_jsonl_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, total=total_rows, desc=f"Embedding {text_key}"):
            if not line.strip():
                continue

            rec = json.loads(line)

            if rec["row"] != row_cursor:
                raise RuntimeError(f"Row mismatch: expected {row_cursor}, got {rec['row']}")

            text = normalize_edge_text(rec.get(text_key))

            if not text:
                raise RuntimeError(f"Empty text at row {row_cursor}")

            if not batch_texts:
                batch_start_row = row_cursor

            batch_texts.append(text)
            row_cursor += 1

            if len(batch_texts) >= batch_size:
                write_batch(batch_texts, batch_start_row)
                batch_texts = []

        if batch_texts:
            write_batch(batch_texts, batch_start_row)

    if row_cursor != total_rows:
        raise RuntimeError(f"Encoded row count mismatch: {row_cursor} != {total_rows}")

    embeddings_memmap.flush()

    print("Saved embeddings:", embeddings_npy_path)
    print("Rows:", total_rows)
    print("Shape:", embeddings_memmap.shape)
    print("Dtype:", embeddings_memmap.dtype)

    del embeddings_memmap
    torch.cuda.empty_cache()
    gc.collect()

In [7]:
#7
# Build relation_id_to_row.jsonl and relation_embeddings.npy.
from datetime import datetime, timezone

OVERWRITE_RELATION_OUTPUTS = True

if RELATION_ID_TO_ROW_JSONL_LOCAL.exists() and not OVERWRITE_RELATION_OUTPUTS:
    print("Relation mapping already exists. Skipping catalog build:", RELATION_ID_TO_ROW_JSONL_LOCAL)
else:
    relation_id = 0
    skipped_relations = 0

    with open(RELATION_ID_TO_ROW_JSONL_LOCAL, "w", encoding="utf-8") as out_f:
        for item in tqdm(iter_kg_items(), desc="Building relation catalog"):
            input_index = item.get("input_index")
            chunk_id = item.get("chunk_id") or item.get("Chunk_id")
            title = item.get("title") or item.get("Title")
            relations = item.get("relations") or []

            for local_relation_index, rel in enumerate(relations):
                if not isinstance(rel, dict):
                    skipped_relations += 1
                    continue

                relation_text = normalize_edge_text(rel.get("relation"))

                if not relation_text:
                    skipped_relations += 1
                    continue

                head_original = rel.get("head_original", rel.get("head"))
                tail_original = rel.get("tail_original", rel.get("tail"))

                # If using normalized JSONL, head/tail are already normalized.
                # If using clean JSON, normalize them here for KG compatibility.
                head_norm = rel.get("head")
                tail_norm = rel.get("tail")

                if not USE_NORMALIZED_JSONL:
                    head_norm = normalize_entity_text(head_original)
                    tail_norm = normalize_entity_text(tail_original)

                record = {
                    "relation_id": relation_id,
                    "row": relation_id,
                    "edge_type": "relation",
                    "input_index": input_index,
                    "chunk_id": chunk_id,
                    "title": title,
                    "local_relation_index": local_relation_index,
                    "head": head_norm,
                    "head_original": head_original,
                    "tail": tail_norm,
                    "tail_original": tail_original,
                    "relation": relation_text,
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                relation_id += 1

    print("Saved relation mapping:", RELATION_ID_TO_ROW_JSONL_LOCAL)
    print("Total relation rows:", relation_id)
    print("Skipped relations:", skipped_relations)

# Encode relation texts.
encode_catalog_jsonl_to_npy(
    catalog_jsonl_path=RELATION_ID_TO_ROW_JSONL_LOCAL,
    embeddings_npy_path=RELATION_EMBEDDINGS_NPY_LOCAL,
    text_key="relation",
    expected_dim=EMBEDDING_DIM,
    batch_size=EMBED_BATCH_SIZE,
    overwrite=OVERWRITE_RELATION_OUTPUTS,
)

# Verify relation outputs.
verify_embedding_output(
    catalog_path=RELATION_ID_TO_ROW_JSONL_LOCAL,
    embeddings_path=RELATION_EMBEDDINGS_NPY_LOCAL,
    id_key="relation_id",
)

# Copy relation outputs to Drive.
copy_output_to_drive(RELATION_ID_TO_ROW_JSONL_LOCAL, RELATION_ID_TO_ROW_JSONL_DRIVE)
copy_output_to_drive(RELATION_EMBEDDINGS_NPY_LOCAL, RELATION_EMBEDDINGS_NPY_DRIVE)

print("\nFinal relation outputs:")
print("Mapping:", RELATION_ID_TO_ROW_JSONL_DRIVE)
print("Embeddings:", RELATION_EMBEDDINGS_NPY_DRIVE)

Building relation catalog: 0it [00:00, ?it/s]

Saved relation mapping: /content/kg_construct_work/hotpotqa/relation_id_to_row.jsonl
Total relation rows: 692402
Skipped relations: 0


Embedding relation:   0%|          | 0/692402 [00:00<?, ?it/s]

Saved embeddings: /content/kg_construct_work/hotpotqa/relation_embeddings.npy
Rows: 692402
Shape: (692402, 4096)
Dtype: float32
Catalog rows: 692402
Embedding shape: (692402, 4096)
Embedding dtype: float32
Norm min/mean/max: 0.9963428378105164 1.001212477684021 1.003891110420227
Verification passed.
Copied: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_id_to_row.jsonl
Copied: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_embeddings.npy

Final relation outputs:
Mapping: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_id_to_row.jsonl
Embeddings: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_embeddings.npy


In [8]:
#8
# Build fact_id_to_row.jsonl and fact_embeddings.npy.
from datetime import datetime, timezone

OVERWRITE_FACT_OUTPUTS = True

if FACT_ID_TO_ROW_JSONL_LOCAL.exists() and not OVERWRITE_FACT_OUTPUTS:
    print("Fact mapping already exists. Skipping catalog build:", FACT_ID_TO_ROW_JSONL_LOCAL)
else:
    fact_id = 0
    skipped_facts = 0

    with open(FACT_ID_TO_ROW_JSONL_LOCAL, "w", encoding="utf-8") as out_f:
        for item in tqdm(iter_kg_items(), desc="Building fact catalog"):
            input_index = item.get("input_index")
            chunk_id = item.get("chunk_id") or item.get("Chunk_id")
            title = item.get("title") or item.get("Title")
            facts = item.get("facts") or []

            for local_fact_index, fact in enumerate(facts):
                if not isinstance(fact, dict):
                    skipped_facts += 1
                    continue

                info_text = normalize_edge_text(fact.get("info"))

                if not info_text:
                    skipped_facts += 1
                    continue

                entity_original = fact.get("entity_original", fact.get("entity"))

                # If using normalized JSONL, entity is already normalized.
                # If using clean JSON, normalize it here for KG compatibility.
                entity_norm = fact.get("entity")

                if not USE_NORMALIZED_JSONL:
                    entity_norm = normalize_entity_text(entity_original)

                record = {
                    "fact_id": fact_id,
                    "row": fact_id,
                    "edge_type": "fact",
                    "input_index": input_index,
                    "chunk_id": chunk_id,
                    "title": title,
                    "local_fact_index": local_fact_index,
                    "entity": entity_norm,
                    "entity_original": entity_original,
                    "info": info_text,
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                fact_id += 1

    print("Saved fact mapping:", FACT_ID_TO_ROW_JSONL_LOCAL)
    print("Total fact rows:", fact_id)
    print("Skipped facts:", skipped_facts)

# Encode fact info texts.
encode_catalog_jsonl_to_npy(
    catalog_jsonl_path=FACT_ID_TO_ROW_JSONL_LOCAL,
    embeddings_npy_path=FACT_EMBEDDINGS_NPY_LOCAL,
    text_key="info",
    expected_dim=EMBEDDING_DIM,
    batch_size=EMBED_BATCH_SIZE,
    overwrite=OVERWRITE_FACT_OUTPUTS,
)

# Verify fact outputs.
verify_embedding_output(
    catalog_path=FACT_ID_TO_ROW_JSONL_LOCAL,
    embeddings_path=FACT_EMBEDDINGS_NPY_LOCAL,
    id_key="fact_id",
)

# Copy fact outputs to Drive.
copy_output_to_drive(FACT_ID_TO_ROW_JSONL_LOCAL, FACT_ID_TO_ROW_JSONL_DRIVE)
copy_output_to_drive(FACT_EMBEDDINGS_NPY_LOCAL, FACT_EMBEDDINGS_NPY_DRIVE)

print("\nFinal fact outputs:")
print("Mapping:", FACT_ID_TO_ROW_JSONL_DRIVE)
print("Embeddings:", FACT_EMBEDDINGS_NPY_DRIVE)

Building fact catalog: 0it [00:00, ?it/s]

Saved fact mapping: /content/kg_construct_work/hotpotqa/fact_id_to_row.jsonl
Total fact rows: 559959
Skipped facts: 0


Embedding info:   0%|          | 0/559959 [00:00<?, ?it/s]

Saved embeddings: /content/kg_construct_work/hotpotqa/fact_embeddings.npy
Rows: 559959
Shape: (559959, 4096)
Dtype: float32
Catalog rows: 559959
Embedding shape: (559959, 4096)
Embedding dtype: float32
Norm min/mean/max: 0.996393084526062 1.0013600587844849 1.003902554512024
Verification passed.
Copied: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_id_to_row.jsonl
Copied: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_embeddings.npy

Final fact outputs:
Mapping: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_id_to_row.jsonl
Embeddings: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_embeddings.npy


In [9]:
#9
# Final verification for all requested outputs.
expected_outputs = {
    "relation_embeddings": RELATION_EMBEDDINGS_NPY_DRIVE,
    "fact_embeddings": FACT_EMBEDDINGS_NPY_DRIVE,
    "relation_id_to_row": RELATION_ID_TO_ROW_JSONL_DRIVE,
    "fact_id_to_row": FACT_ID_TO_ROW_JSONL_DRIVE,
}

for name, path in expected_outputs.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{name}: {path} | {size_mb:.2f} MB")

print("\nRelation verification:")
verify_embedding_output(
    catalog_path=RELATION_ID_TO_ROW_JSONL_DRIVE,
    embeddings_path=RELATION_EMBEDDINGS_NPY_DRIVE,
    id_key="relation_id",
)

print("\nFact verification:")
verify_embedding_output(
    catalog_path=FACT_ID_TO_ROW_JSONL_DRIVE,
    embeddings_path=FACT_EMBEDDINGS_NPY_DRIVE,
    id_key="fact_id",
)

print("\nAll four requested files are valid.")

relation_embeddings: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_embeddings.npy | 10818.78 MB
fact_embeddings: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_embeddings.npy | 8749.36 MB
relation_id_to_row: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/relation_id_to_row.jsonl | 252.90 MB
fact_id_to_row: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/fact_id_to_row.jsonl | 178.75 MB

Relation verification:
Catalog rows: 692402
Embedding shape: (692402, 4096)
Embedding dtype: float32
Norm min/mean/max: 0.9963428378105164 1.001212477684021 1.003891110420227
Verification passed.

Fact verification:
Catalog rows: 559959
Embedding shape: (559959, 4096)
Embedding dtype: float32
Norm min/mean/max: 0.996393084526062 1.0013600587844849 1.003902554512024
Verification passed.

All four requested files are valid.
